In [2]:
import duckdb
import os
from pathlib import Path

In [4]:
# ensuring that os.chdir is idempotent and that we are in the project root directory,
# not inside the notebooks directory
if 'notebooks' not in os.listdir(Path.cwd()):
    print("Still inside notebooks directory, changing to project root directory.")
    os.chdir(Path.cwd().parent)
    print("Current working directory after change: ", Path.cwd())
else:
    print(f"Already in parent directory (current working directory: {Path.cwd()})")


Already in parent directory (current working directory: d:\space-weather-project-scrub)


In [5]:
from src.io.load_config import load_config
sw_config = load_config()['space_weather']

In [6]:
sw_config.keys()

dict_keys(['base_url', 'api_key_env', 'date_fmt', 'ingestion', 'preprocessing', 'endpoints', 'defaults', 'allowed_values', 'api_key'])

In [7]:
t1_path = sw_config['preprocessing']['k_index']['T1_output_dir']
t1_path

'data/02-preprocessed/space_weather/k_index/T1'

# first inpsect number of unique runs per `(valid_time, location) = assumed to be the identifier for a kindex obs in the real world`


In [8]:
# number of unique runs and analysis times per valid_time and location

duckdb.sql(
    f"""
    SELECT valid_time,
        location,
        COUNT(DISTINCT run_id) AS num_unique_runs,
        COUNT(DISTINCT analysis_time) AS num_unique_analysis_times
    FROM read_parquet('{t1_path}')
    GROUP BY valid_time, location
    HAVING num_unique_runs > 1
    ORDER BY num_unique_runs DESC
    """.strip()
)


┌─────────────────────┬───────────────────┬─────────────────┬───────────────────────────┐
│     valid_time      │     location      │ num_unique_runs │ num_unique_analysis_times │
│      timestamp      │      varchar      │      int64      │           int64           │
├─────────────────────┼───────────────────┼─────────────────┼───────────────────────────┤
│ NULL                │ Australian region │               3 │                         0 │
│ 2025-03-15 00:00:00 │ Australian region │               2 │                         1 │
│ 2023-01-15 18:00:00 │ Australian region │               2 │                         1 │
│ 2023-01-26 18:00:00 │ Australian region │               2 │                         1 │
│ 2025-09-10 21:00:00 │ Australian region │               2 │                         1 │
│ 2018-01-13 06:00:00 │ Australian region │               2 │                         1 │
│ 2025-03-08 03:00:00 │ Australian region │               2 │                         1 │
│ 2023-01-

# can confirm: just take latest runid, analysis_time already unified for each (valid_time, location)

In [ ]:
duckdb.sql(
    f"""
    SELECT valid_time,
        location,
        MAX(run_id) AS latest_run_id,
        COUNT(DISTINCT kindex) > 1 AS flag_inconsistent_kindex

        FROM read_parquet('{t1_path}')

        GROUP BY valid_time, location
    """
)

┌─────────────────────┬───────────────────┬──────────────────┬──────────────────────────┐
│     valid_time      │     location      │  latest_run_id   │ flag_inconsistent_kindex │
│      timestamp      │      varchar      │     varchar      │         boolean          │
├─────────────────────┼───────────────────┼──────────────────┼──────────────────────────┤
│ 2025-03-02 09:00:00 │ Australian region │ 20260307T050221Z │ false                    │
│ 2025-03-04 00:00:00 │ Australian region │ 20260307T050221Z │ false                    │
│ 2025-03-05 09:00:00 │ Australian region │ 20260307T050221Z │ false                    │
│ 2025-03-05 12:00:00 │ Australian region │ 20260307T050221Z │ false                    │
│ 2025-03-08 06:00:00 │ Australian region │ 20260307T050221Z │ false                    │
│ 2025-03-08 12:00:00 │ Australian region │ 20260307T050221Z │ false                    │
│ 2025-03-11 15:00:00 │ Australian region │ 20260307T050221Z │ false                    │
│ 2025-03-

In [9]:
# for each (valid_time, location), obtain kindex corresponding to latest runid

# 1. obtain latest runid first,
# 2. then filter to get kindex values for those latest runids
# 3. then join back to get kindex values for those runids
df_transform = duckdb.sql(
    f"""

    WITH
    latest_runs AS (
        SELECT valid_time,
               location,
               MAX(run_id) AS latest_run_id,
               COUNT(DISTINCT kindex) > 1 AS flag_inconsistent_kindex

        FROM read_parquet('{t1_path}')

        GROUP BY valid_time, location
    ),

    kindex_latest AS (
        SELECT kindex_data.valid_time,
        kindex_data.location,
        kindex_data.kindex,
        latest_runs.latest_run_id,
        latest_runs.flag_inconsistent_kindex

        FROM

        read_parquet('{t1_path}') AS kindex_data

        INNER JOIN
        latest_runs
            ON kindex_data.valid_time = latest_runs.valid_time 
            AND kindex_data.location = latest_runs.location
            AND kindex_data.run_id = latest_runs.latest_run_id
    )

    SELECT *
    FROM kindex_latest
    ORDER BY flag_inconsistent_kindex DESC;

    """.strip()
    
).to_df()

display(df_transform)

display(
    df_transform.\
    groupby(['location', 'valid_time']).\
    size().\
    reset_index(name='count').\
    sort_values('count', ascending=False)
)


,valid_time,location,kindex,latest_run_id,flag_inconsistent_kindex
0,2025-06-30 00:00:00,Alice Springs,1,20260312T005243Z,False
1,2025-06-30 03:00:00,Alice Springs,2,20260312T005243Z,False
2,2025-06-30 06:00:00,Alice Springs,2,20260312T005243Z,False
3,2025-06-30 09:00:00,Alice Springs,3,20260312T005243Z,False
4,2025-06-30 12:00:00,Alice Springs,1,20260312T005243Z,False
...,...,...,...,...,...
31628,2022-09-11 09:00:00,Australian region,1,20260322T105629Z,False
31629,2022-09-11 12:00:00,Australian region,2,20260322T105629Z,False
31630,2022-09-11 15:00:00,Australian region,3,20260322T105629Z,False
31631,2022-09-11 18:00:00,Australian region,2,20260322T105629Z,False


,location,valid_time,count
0,Alice Springs,2025-01-01 00:00:00,1
21099,Australian region,2023-04-24 09:00:00,1
21097,Australian region,2023-04-24 03:00:00,1
21096,Australian region,2023-04-24 00:00:00,1
21095,Australian region,2023-04-23 21:00:00,1
...,...,...,...
10540,Australian region,2019-09-11 15:00:00,1
10539,Australian region,2019-09-11 12:00:00,1
10538,Australian region,2019-09-11 09:00:00,1
10537,Australian region,2019-09-11 06:00:00,1


In [13]:
1 + "1"

TypeError: unsupported operand type(s) for +: 'int' and 'str'